# deberta + lgbm

In [ ]:
import pandas as pd
import re

def url_to_semantics(text: str) -> str:
    if not isinstance(text, str):
        return ""

    url_pattern = r'https?://[^\s/$.?#].[^\s]*'
    urls = re.findall(url_pattern, text)

    if not urls:
        return ""

    all_semantics = []
    seen_semantics = set()

    for url in urls:
        url_lower = url.lower()

        domain_match = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", url_lower)
        if domain_match:
            full_domain = domain_match.group(1)
            parts = full_domain.split('.')
            for part in parts:
                if part and part not in seen_semantics and len(part) > 3: # Avoid short parts like 'www'
                    all_semantics.append(f"domain:{part}")
                    seen_semantics.add(part)

        # 2. Extract path parts
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", url_lower)
        path_parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()] # Split by common delimiters

        for part in path_parts:
            # Clean up potential file extensions or query params
            part_clean = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", part)
            if part_clean and part_clean not in seen_semantics and len(part_clean) > 3:
                all_semantics.append(f"path:{part_clean}")
                seen_semantics.add(part_clean)

    if not all_semantics:
        return ""

    return f"\nURL Keywords: {' '.join(all_semantics)}"


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []

    flatten.append(train_dataset[["body", "rule", "subreddit","rule_violation"]].copy())

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"

            if col_name in train_dataset.columns:
                sub_dataset = train_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0

                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]

                if not sub_dataset.empty:
                    flatten.append(sub_dataset)

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"

            if col_name in test_dataset.columns:
                sub_dataset = test_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0

                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]

                if not sub_dataset.empty:
                    flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(subset=['body', 'rule', 'subreddit'], ignore_index=True)
    dataframe.drop_duplicates(subset=['body','rule'],keep='first',inplace=True)

    return dataframe.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from typing import Optional

class PairTextDataset(Dataset):
    def __init__(self, rules, bodies, tokenizer, max_length: int):
        self.enc = tokenizer(
            rules, bodies,
            truncation=True, padding=True, max_length=max_length,
            return_token_type_ids=True
        )

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.enc["input_ids"][idx]),
            "attention_mask": torch.tensor(self.enc["attention_mask"][idx]),
            "token_type_ids": torch.tensor(self.enc.get("token_type_ids", [[0]*len(self.enc["input_ids"][idx])])[idx]),
        }

@torch.no_grad()
def extract_features(
    model_path: str, df: pd.DataFrame,
    max_length: int = 256,
    batch_size: int = 16,
    device: Optional[str] = None
) -> np.ndarray:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_path)
    ds = PairTextDataset(df["rule"].tolist(), df["body_with_url"].tolist(), tok, max_length)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    backbone = AutoModel.from_pretrained(model_path, output_hidden_states=True).to(device).eval()

    feats = []
    for batch in dl:
        batch = {k: v.to(device) for k, v in batch.items() if v is not None}
        outputs = backbone(**batch, output_hidden_states=True, return_dict=True)
        hs = outputs.hidden_states  # tuple(len=layers+1)
        # last 4 layers, CLS token, mean over layers -> [B, H]
        cls_stack = torch.stack([hs[-i][:, 0, :] for i in range(1, 5)], dim=0).mean(dim=0)
        feats.append(cls_stack.detach().cpu().numpy())
    return np.concatenate(feats, axis=0)

def train_deberta_lgbm_model(
    training_df: pd.DataFrame,
    model_path: str = "/kaggle/input/huggingfacedebertav3variants/deberta-v3-large",
    max_length: int = 256,
    batch_size: int = 4,
    output_filename: str = "submission_deberta_lgbm.csv",
    seed: int = 42
) -> str:
    # 特徴抽出（DeBERTaは固定・非学習）
    X = extract_features(model_path, training_df, max_length=max_length, batch_size=batch_size)
    y = training_df["rule_violation"].astype(int).values

    # バリデーション分割（早期停止用）
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.15, random_state=seed, stratify=y)

    clf = lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1
    )
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
    )

    return clf

In [ ]:
import random
import os
import torch

import numpy as np
import pandas as pd

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CFG:
    model_name_or_path = "/kaggle/input/huggingfacedebertav3variants/deberta-v3-large"
    data_path = "/kaggle/input/jigsaw-agile-community-rules/"
    output_dir = "./deberta_v3_small_final_model"

    EPOCHS = 3
    LEARNING_RATE = 2e-05
    MAX_LENGTH = 256
    BATCH_SIZE = 32

    # Cross-validation settings
    # N_SPLITS = 5  # Number of folds for cross-validation
    RANDOM_STATE = 42

def main():
    seed_everything(CFG.RANDOM_STATE)

    full_df = get_dataframe_to_train(CFG.data_path)
    full_df['body_with_url'] = full_df['body'].apply(lambda x: x + url_to_semantics(x))

    # train_df, val_df = train_test_split(full_df, test_size=0.2, random_state=seed, stratify=y)
    test_df = pd.read_csv(f"{CFG.data_path}/test.csv")

    model = train_deberta_lgbm_model(full_df)

    # predict
    X_test = extract_features(model_path, test_df, max_length=CFG.MAX_LENGTH, batch_size=CFG.BATCH_SIZE)
    probs = model.predict_proba(X_test)[:, 1]
    pd.DataFrame({"row_id": test_df["row_id"], "rule_violation": probs}).to_csv("submission.csv", index=False)


# cv

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

def cv_lgbm_auc(X, y, n_splits=5, seed=42, early_stopping=200, verbose=100, **params):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    val_aucs, train_aucs, best_iters = [], [], []
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        clf = lgb.LGBMClassifier(
            objective="binary",
            boosting_type="gbdt",
            n_estimators=5000,
            learning_rate=0.05,
            # num_leaves=63,
            # max_depth=-1,
            # min_data_in_leaf=64,
            # feature_fraction=0.9,
            # bagging_fraction=0.8,
            # bagging_freq=1,
            # lambda_l2=0.0,
            random_state=seed,
            n_jobs=-1,
            **params
        )
        clf.fit(
            X[tr], y[tr],
            eval_set=[(X[va], y[va]), (X[tr], y[tr])],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(early_stopping, verbose=False),
                lgb.log_evaluation(verbose)
            ]
        )
        best_iter = clf.best_iteration_
        p_val = clf.predict_proba(X[va], num_iteration=best_iter)[:, 1]
        p_tr  = clf.predict_proba(X[tr], num_iteration=best_iter)[:, 1]
        auc_v = roc_auc_score(y[va], p_val)
        auc_t = roc_auc_score(y[tr], p_tr)
        val_aucs.append(auc_v); train_aucs.append(auc_t); best_iters.append(best_iter)
        gap = auc_t - auc_v
        print(f"fold{fold}: val_auc={auc_v:.4f}, train_auc={auc_t:.4f}, gap={gap:.4f}, best_iter={best_iter}")
    print(f"CV val_auc: mean={np.mean(val_aucs):.4f} ± {np.std(val_aucs):.4f}")
    print(f"gap(mean train - val): {np.mean(np.array(train_aucs)-np.array(val_aucs)):.4f}")
    print(f"best_iter median: {int(np.median(best_iters))}")
    return {"val_auc": val_aucs, "train_auc": train_aucs, "best_iter": best_iters}

In [ ]:
import random
import os
import torch

import numpy as np
import pandas as pd

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CFG:
    model_name_or_path = "/kaggle/input/huggingfacedebertav3variants/deberta-v3-large"
    data_path = "/kaggle/input/jigsaw-agile-community-rules/"
    output_dir = "./deberta_v3_small_final_model"

    EPOCHS = 3
    LEARNING_RATE = 2e-05
    MAX_LENGTH = 256
    BATCH_SIZE = 32

    # Cross-validation settings
    # N_SPLITS = 5  # Number of folds for cross-validation
    RANDOM_STATE = 42

def main():
    seed_everything(CFG.RANDOM_STATE)

    full_df = get_dataframe_to_train(CFG.data_path)
    full_df['body_with_url'] = full_df['body'].apply(lambda x: x + url_to_semantics(x))

    # train_df, val_df = train_test_split(full_df, test_size=0.2, random_state=seed, stratify=y)

    # 特徴抽出（DeBERTaは固定・非学習）
    X = extract_features(CFG.model_name_or_path, full_df, max_length=CFG.MAX_LENGTH, batch_size=CFG.BATCH_SIZE)
    y = full_df["rule_violation"].astype(int).values

    best = cv_lgbm_auc(X, y)

    # predict
    # test_df = pd.read_csv(f"{CFG.data_path}/test.csv")
    # test_df['body_with_url'] = test_df['body'].apply(lambda x: x + url_to_semantics(x))
    # X_test = extract_features(CFG.model_name_or_path, test_df, max_length=CFG.MAX_LENGTH, batch_size=CFG.BATCH_SIZE)

    # probs = model.predict_proba(X_test)[:, 1]
    # pd.DataFrame({"row_id": test_df["row_id"], "rule_violation": probs}).to_csv("submission.csv", index=False)
